# W04A — Cómo “piensa” el motor: **EXPLAIN**, cardinalidad y performance (DuckDB)

**Objetivo:** aprender a **leer planes de ejecución** y detectar los 3 problemas más comunes en analítica:
1) **scan enorme** (lees más de lo que necesitas),
2) **JOIN que infló** (cardinalidad mal controlada),
3) **agregación cara** (filtro tarde / columnas de más).

## Bibliografía (W04 en general)

### Libro guía — **DDIA (Martin Kleppmann)**
- **Capítulo 3: Storage and Retrieval**  
  Úsalo para: *OLTP vs OLAP, por qué en analítica importa el ancho de banda, column-oriented storage, y por qué el layout/índices cambian el costo de las consultas.*


### Documentación práctica — DuckDB
- `EXPLAIN` / `EXPLAIN ANALYZE` (planes y cardinalidad)
- Profiling con `PRAGMA enable_profiling`
- Índices (`CREATE INDEX`) y tipos de índices (zonemap / ART)


In [1]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path(".").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
ART_DIR = PROJECT_ROOT / "artifacts"
DOCS_DIR = PROJECT_ROOT / "docs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH))

raw_csv = RAW_DIR / "pscomppars.csv"
if not raw_csv.exists():
    raise FileNotFoundError(
        f"No encuentro {raw_csv}. Necesitas el CSV de W01/W02."
    )

def sql_quote(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"

con.execute(f'''
CREATE OR REPLACE VIEW raw_ps AS
SELECT * FROM read_csv_auto({sql_quote(str(raw_csv.resolve()))})
''')

# Si ya tienes Silver/Facts de W03B, perfecto. Si no, los construimos rápido aquí (idéntico a W03B).
con.execute("DROP TABLE IF EXISTS silver_planet")
con.execute('''
CREATE TABLE silver_planet AS
SELECT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  sy_snum,
  sy_pnum,
  sy_dist,
  ra,
  dec,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt,
  st_teff,
  st_rad,
  st_mass
FROM raw_ps
WHERE pl_name IS NOT NULL
  AND hostname IS NOT NULL
  AND (disc_year IS NULL OR (disc_year BETWEEN 1980 AND 2026))
  AND (pl_rade  IS NULL OR (pl_rade  > 0 AND pl_rade  <= 30))
  AND (pl_bmasse IS NULL OR (pl_bmasse > 0))
''')

con.execute("DROP TABLE IF EXISTS dim_host_full")
con.execute('''
CREATE TABLE dim_host_full AS
SELECT
  hostname,
  MAX(sy_dist)  AS sy_dist,
  MAX(ra)       AS ra,
  MAX(dec)      AS dec,
  MAX(st_teff)  AS st_teff,
  MAX(st_rad)   AS st_rad,
  MAX(st_mass)  AS st_mass
FROM silver_planet
GROUP BY hostname
''')

con.execute("DROP TABLE IF EXISTS fact_planet")
con.execute('''
CREATE TABLE fact_planet AS
SELECT DISTINCT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt
FROM silver_planet
''')

con.sql("SELECT COUNT(*) AS n_fact, COUNT(DISTINCT pl_name) AS n_pl, COUNT(DISTINCT hostname) AS n_hosts FROM fact_planet").show()

┌────────┬───────┬─────────┐
│ n_fact │ n_pl  │ n_hosts │
│ int64  │ int64 │  int64  │
├────────┼───────┼─────────┤
│   6101 │  6101 │    4550 │
└────────┴───────┴─────────┘



## 1) Qué vamos a mirar en un plan (lectura rápida)

Cuando mires `EXPLAIN`, busca:
- **SCAN**: ¿está leyendo demasiadas filas/columnas?
- **FILTER**: ¿se aplica temprano o tarde?
- **JOIN**: ¿qué tipo? (HASH JOIN típico en analítica)
- **GROUP BY**: ¿cuántas filas entran?
- **CARDINALIDAD**: cuántas filas pasan por cada operador

`EXPLAIN` = estimado  
`EXPLAIN ANALYZE` = real (ejecuta y muestra cardinalidad real + tiempos)

In [2]:
# TU TURNO 1: pega aquí una consulta tipo "métrica" y mira su plan.
# Debe tener: WHERE + GROUP BY (como en W02A).
# Ejemplo de idea: por disc_year o por discoverymethod.

q = '''
SELECT
    disc_year,
    COUNT(*) AS n_planets,
    AVG(pl_rade) AS avg_radius_earth,
    AVG(pl_bmasse) AS avg_mass_jupiter
FROM silver_planet
WHERE disc_year BETWEEN 2000 AND 2025
  AND pl_rade IS NOT NULL
  AND pl_bmasse IS NOT NULL
GROUP BY disc_year
ORDER BY disc_year
'''

con.sql("EXPLAIN " + q).show()
# Si corre rápido (esperamos que sí, porque silver_planet no es enorme):
con.sql("EXPLAIN ANALYZE " + q).show()

┌───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [3]:
# TU TURNO 2: escribe dos consultas equivalentes:
# A) con SELECT * y B) con solo 3 columnas.
# Luego compara sus planes.

qA = '''
SELECT *
FROM silver_planet
WHERE disc_year > 2010
  AND discoverymethod IS NOT NULL
'''

qB = '''
SELECT pl_name, hostname, disc_year
FROM silver_planet
WHERE disc_year > 2010
  AND discoverymethod IS NOT NULL
'''

print("=== Plan con SELECT * ===")
con.sql("EXPLAIN " + qA).show()

print("\n=== Plan con solo 3 columnas ===")
con.sql("EXPLAIN " + qB).show()

=== Plan con SELECT * ===
┌───────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [4]:
# TU TURNO 3: valida un JOIN sano con evidencia
# 1) arma un JOIN fact_planet + dim_host_full (LEFT JOIN)
# 2) muestra EXPLAIN
# 3) muestra n_fact vs n_join

q_join = '''
SELECT
    f.pl_name,
    f.hostname,
    f.discoverymethod,
    f.disc_year,
    d.sy_dist,
    d.st_teff,
    d.st_mass
FROM fact_planet f
LEFT JOIN dim_host_full d ON f.hostname = d.hostname
WHERE f.disc_year >= 2000
  AND f.pl_rade IS NOT NULL
'''

con.sql("EXPLAIN " + q_join).show()

n_fact = con.sql("SELECT COUNT(*) FROM fact_planet").fetchone()[0]
n_join = con.sql(f'''
SELECT COUNT(*) FROM (
    {q_join}
) AS joined
''').fetchone()[0]

print(f"Filas en fact_planet: {n_fact}")
print(f"Filas después del LEFT JOIN: {n_join}")
print("¿Coinciden? →", n_fact == n_join)

┌───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## Para entregar (W04A) — mesurado

### En clase
1) `docs/w04a_perf_report.md` con:
   - 2 planes (`EXPLAIN`) pegados (consulta tuya + JOIN)
   - 2 conclusiones cortas (2–3 líneas cada una) sobre:
     - dónde está el costo (SCAN/JOIN/GROUP BY)
     - qué mejorarías (leer menos columnas, filtrar antes, etc.)
2) `docs/decisions_log.md`: 1 decisión:
   - “Reescribí una consulta para reducir costo” + evidencia (plan antes/después o conteo).

### Tarea
1) Ejecuta `EXPLAIN ANALYZE` sobre 1 consulta y pega el resultado.
2) Exporta a `artifacts/` un archivo de evidencia:
   - `artifacts/w04a_explain_q1.txt` (copiar/pegar plan a texto está bien)

## Reflexión (bitácora)
- ¿Qué parte del plan te costó más interpretar?
- Si el dataset creciera 100×, ¿qué operador crees que empeora primero (SCAN/JOIN/GROUP BY) y por qué?

## Solución Entregable

In [5]:
q_metric = '''
SELECT
    discoverymethod,
    COUNT(*) AS n_planets,
    AVG(pl_rade) AS avg_radius
FROM silver_planet
WHERE disc_year BETWEEN 2010 AND 2025
  AND discoverymethod IS NOT NULL
  AND pl_rade IS NOT NULL
GROUP BY discoverymethod
ORDER BY n_planets DESC
'''

plan_metric = con.execute('EXPLAIN ' + q_metric).fetchall()
plan_metric


[('physical_plan',
  '┌───────────────────────────┐\n│          ORDER_BY         │\n│    ────────────────────   │\n│     count_star() DESC     │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│       HASH_GROUP_BY       │\n│    ────────────────────   │\n│         Groups: #0        │\n│                           │\n│        Aggregates:        │\n│        count_star()       │\n│          avg(#1)          │\n│                           │\n│          ~9 rows          │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│         PROJECTION        │\n│    ────────────────────   │\n│      discoverymethod      │\n│          pl_rade          │\n│                           │\n│        ~1,220 rows        │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│          SEQ_SCAN         │\n│    ────────────────────   │\n│           Table:          │\n│      exoplanets.main      │\n│       .silver_planet      │\n│                           │\n│   Type: Sequent

In [6]:
q_join_perf = '''
SELECT
    f.pl_name,
    f.hostname,
    f.discoverymethod,
    d.sy_dist,
    d.st_mass
FROM fact_planet f
LEFT JOIN dim_host_full d
  ON f.hostname = d.hostname
WHERE f.disc_year >= 2010
  AND f.pl_bmasse IS NOT NULL
'''

plan_join = con.execute('EXPLAIN ' + q_join_perf).fetchall()
plan_join


[('physical_plan',
  '┌───────────────────────────┐\n│         PROJECTION        │\n│    ────────────────────   │\n│          pl_name          │\n│          hostname         │\n│      discoverymethod      │\n│          sy_dist          │\n│          st_mass          │\n│                           │\n│        ~4,550 rows        │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│         HASH_JOIN         │\n│    ────────────────────   │\n│      Join Type: RIGHT     │\n│                           │\n│        Conditions:        ├──────────────┐\n│    hostname = hostname    │              │\n│                           │              │\n│        ~4,550 rows        │              │\n└─────────────┬─────────────┘              │\n┌─────────────┴─────────────┐┌─────────────┴─────────────┐\n│          SEQ_SCAN         ││          SEQ_SCAN         │\n│    ────────────────────   ││    ────────────────────   │\n│           Table:          ││           Table:          │\n│      exopl

In [7]:
analyze_metric = con.execute('EXPLAIN ANALYZE ' + q_metric).fetchall()
analyze_metric


[('analyzed_plan',
  '┌─────────────────────────────────────┐\n│┌───────────────────────────────────┐│\n││    Query Profiling Information    ││\n│└───────────────────────────────────┘│\n└─────────────────────────────────────┘\nEXPLAIN ANALYZE  SELECT     discoverymethod,     COUNT(*) AS n_planets,     AVG(pl_rade) AS avg_radius FROM silver_planet WHERE disc_year BETWEEN 2010 AND 2025   AND discoverymethod IS NOT NULL   AND pl_rade IS NOT NULL GROUP BY discoverymethod ORDER BY n_planets DESC \n┌────────────────────────────────────────────────┐\n│┌──────────────────────────────────────────────┐│\n││              Total Time: 0.0095s             ││\n│└──────────────────────────────────────────────┘│\n└────────────────────────────────────────────────┘\n┌───────────────────────────┐\n│           QUERY           │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│      EXPLAIN_ANALYZE      │\n│    ────────────────────   │\n│                           │\n│           0 rows       

### Exportación de evidencia

Guardo el texto completo del `EXPLAIN ANALYZE` para que quede una evidencia reutilizable fuera del notebook.


In [8]:
ART_DIR.mkdir(parents=True, exist_ok=True)
explain_path = ART_DIR / 'w04a_explain_q1.txt'

with open(explain_path, 'w', encoding='utf-8') as f:
    f.write('EXPLAIN q_metric\n')
    for row in plan_metric:
        f.write(str(row) + '\n')

    f.write('\nEXPLAIN q_join_perf\n')
    for row in plan_join:
        f.write(str(row) + '\n')

    f.write('\nEXPLAIN ANALYZE q_metric\n')
    for row in analyze_metric:
        f.write(str(row) + '\n')

print('Archivo exportado en:', explain_path)


Archivo exportado en: /home/manuela-rios/Documentos/Física computacional/Manuela-Rios/notebooks/artifacts/w04a_explain_q1.txt
